# X-Mobility 재평가 · 체크포인트 평균 — Kaggle (E · 서진석)

여러 ckpt 를 **같은 test** 에서 `e2e/eval_offline.py` 로 평가하고, 같은 사전학습 ckpt 에서 미세조정한 ckpt 들의 가중치 평균(model soup, `e2e/avg_ckpt.py`)도 만들어 평가한다. 학습이 없어 **HF 토큰이 필요 없다**(CLI `kaggle kernels push` 로 바로 돌릴 수 있음).

- Input: 평가 데이터셋(기본 `dataset-03-xm-v1-2` — test · val 은 v1.1 분할) + ckpt 데이터셋(`fd-xm-ckpts`, 파일 이름 = 아래 `CKPTS`).
- 출력: `eval_<태그>_<split>/`(summary.json · frames.csv · run_meta.json) · `soups.json`(평균 ckpt sha) · `eval_index.json`(한 줄 요약) · `logs/`. 평균 ckpt 자체는 /kaggle/tmp 에만(같은 입력이면 다시 만들어도 똑같음).
- 받은 뒤: 로컬에서 `steer_diag.py` · `rollout_drift.py` 로 진단(`e2e/results/compare_*.md`).


In [ ]:
CKPTS = {                                  # 태그: ckpt 데이터셋 안 파일 이름 (sha12 는 평가 run_meta 의 checkpoint_sha12 로 확인)
    "v0.1": "xm_ft_v0.1_e8.ckpt",          # d86970ce1faf · xm_v1 · 명령 가중치 10
    "aw30": "xm_ft_v0.1_aw30_e8.ckpt",     # 4fdf249cda0a · xm_v1 · 명령 가중치 30 (원래 파일 이름이 v0.1 과 같아 바꿔 올림)
    "v0.2": "xm_ft_v0.2_e9.ckpt",          # xm_v1.2 (v1.1 + Nav2 train x3)
}
SOUPS = {                                  # 균등 평균. 모두 nvidia/X-Mobility 에서 lr 1e-5 로 미세조정 → 같은 골짜기 가정(Wortsman 2022)
    "soup_v01_v02": ["v0.1", "v0.2"],
    "soup_v01_aw30": ["v0.1", "aw30"],
    "soup_all3": ["v0.1", "aw30", "v0.2"],
}
EVAL_TAGS = ["v0.1", "aw30", "soup_v01_v02", "soup_v01_aw30", "soup_all3"]   # v0.2 · 제로샷의 v1.1 test 는 v0.2 학습 버전 출력에 있음
SPLITS = ["test"]
REPO_URL = "https://github.com/JP-Avengers/FUTURE-DRIVE-PROJECT.git"
XM_URL, XM_REV = "https://github.com/NVlabs/X-Mobility.git", "5837aea"
WORK = "/kaggle/working"


In [ ]:
import os, subprocess, glob, json, math, shutil, time
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise SystemExit(f"실패({r.returncode}): {cmd}")
    return r.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
os.chdir("/kaggle/tmp" if os.path.isdir("/kaggle/tmp") else "/tmp")
if not os.path.isdir("FUTURE-DRIVE-PROJECT"):
    sh(f"git clone --depth 20 {REPO_URL}")
if not os.path.isdir("X-Mobility"):
    sh(f"git clone {XM_URL} && cd X-Mobility && git checkout -q {XM_REV}")
REPO, XM = os.path.abspath("FUTURE-DRIVE-PROJECT"), os.path.abspath("X-Mobility")
sh(f"git -C {REPO} log --oneline -1 && git -C {XM} log --oneline -1")
assert "GIN_SKIP" in open(f"{REPO}/e2e/finetune.py").read(), "레포 main 에 finetune.py 새 판(gin 파싱 수정)이 아직 없음 — push 먼저"
assert os.path.exists(f"{REPO}/e2e/avg_ckpt.py"), "레포 main 에 avg_ckpt.py 없음"


In [ ]:
os.environ["USE_TF"] = "0"  # wandb 0.19.4 가 protobuf 를 5.29 로 내려 Kaggle 의 TF(≥6.31 필요)를 불러오면 죽음 → transformers 가 TF 를 안 보게
# X-Mobility 가 맞춘 판(requirements.txt)과 같게. torch 는 Kaggle 기본(CUDA) 그대로.
# huggingface_hub 는 transformers 4.48.1 범위(<1.0) 안에서 0.25 로 고정. wandb 는 X-Mobility trainer 가 import.
sh("pip install -q gin-config==0.5.0 einops==0.7.0 pytorch-lightning==2.5.0.post0 transformers==4.48.1 "
   "diffusers==0.29.2 huggingface_hub==0.25.2 timm==1.0.14 torcheval==0.0.7 torchmetrics wandb==0.19.4")
sh("python -c \"import torch, transformers, diffusers, pytorch_lightning as pl; "
   "from diffusers import UNet2DConditionModel; from transformers import CLIPTextModel; "
   "print(torch.__version__, torch.cuda.device_count(), transformers.__version__, diffusers.__version__, pl.__version__)\"")

In [ ]:
cand = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train", recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), "val")) and os.path.isdir(os.path.join(os.path.dirname(p), "test"))]
assert cand, "입력 데이터셋에서 train/val/test 를 못 찾음"
DATA = sorted(cand, key=len)[0]
print("DATA =", DATA)
assert not os.path.exists(f"{DATA}/PATH_SCALE.json"), "경로 축척 데이터 — 원래 축척 ckpt 를 이 데이터로 평가하면 ADE · FDE 가 틀어짐"
found = {os.path.basename(p): p for p in glob.glob("/kaggle/input/**/*.ckpt", recursive=True)}
print("ckpt:", *sorted(found), sep="\n  ")
CK = {t: found[f] for t, f in CKPTS.items() if f in found}
assert len(CK) == len(CKPTS), f"없는 ckpt: {sorted(set(CKPTS) - set(CK))}"


In [ ]:
# 체크포인트 평균 — 실패해도 개별 ckpt 평가는 계속
SOUP_DIR = ("/kaggle/tmp" if os.path.isdir("/kaggle/tmp") else "/tmp") + "/soups"
os.makedirs(SOUP_DIR, exist_ok=True)
soups = {}
for tag, parts in SOUPS.items():
    out = f"{SOUP_DIR}/{tag}.ckpt"
    if not os.path.exists(out):
        r = subprocess.run(f"python {REPO}/e2e/avg_ckpt.py -o {out} " + " ".join(f'"{CK[p]}"' for p in parts),
                           shell=True, capture_output=True, text=True)
        print(r.stdout[-2000:], r.stderr[-2000:], flush=True)
        if r.returncode == 0 and r.stdout.strip():
            soups[tag] = dict(json.loads(r.stdout.strip().splitlines()[-1]), parts=parts)
    if os.path.exists(out):
        CK[tag] = out
json.dump(soups, open(f"{WORK}/soups.json", "w"), indent=1)
print(json.dumps(soups, indent=1))


In [ ]:
# 평가 — GPU 마다 한 개씩 동시에(T4 x 2). 로그는 logs/
import torch
EVAL_BASE = f"python {REPO}/e2e/eval_offline.py --xm-repo {XM} -c {REPO}/e2e/configs/finetune_parking_r3.gin -d {DATA}"
os.makedirs(f"{WORK}/logs", exist_ok=True)
queue = [(t, sp) for t in EVAL_TAGS for sp in SPLITS if t in CK and not os.path.exists(f"{WORK}/eval_{t}_{sp}/summary.json")]
print("평가 못 함(ckpt 없음):", [t for t in EVAL_TAGS if t not in CK])
ngpu = max(1, torch.cuda.device_count())
active = {}
while queue or active:
    for dev in range(ngpu):
        if dev not in active and queue:
            t, sp = queue.pop(0)
            cmd = f'{EVAL_BASE} --split {sp} -p "{CK[t]}" -o {WORK}/eval_{t}_{sp} --device cuda:{dev}'
            print(f"[시작 cuda:{dev}] {t} {sp}\n  $ {cmd}", flush=True)
            active[dev] = (subprocess.Popen(cmd, shell=True, stdout=open(f"{WORK}/logs/eval_{t}_{sp}.log", "w"),
                                            stderr=subprocess.STDOUT), t, sp, time.time())
    time.sleep(15)
    for dev, (p, t, sp, t0) in list(active.items()):
        if p.poll() is not None:
            print(f"[끝 cuda:{dev}] {t} {sp} rc={p.returncode} {time.time() - t0:.0f} s", flush=True)
            if p.returncode:
                print(open(f"{WORK}/logs/eval_{t}_{sp}.log").read()[-3000:])
            del active[dev]


In [ ]:
rows = []
for t in EVAL_TAGS:
    for sp in SPLITS:
        f = f"{WORK}/eval_{t}_{sp}/summary.json"
        if not os.path.exists(f):
            print("결과 없음:", t, sp)
            continue
        s = json.load(open(f)); o = s["overall"]; b = s["by_scenario"]
        a, n = b.get("r3_auto", {}), b.get("r3_nav2_obs", {})
        rows.append(dict(tag=t, split=sp, sha12=s["setup"].get("checkpoint_sha12"), n=o["n"], v_mae=o["v_mae"], w_mae=o["w_mae"],
                         w_sign=o["w_sign_agree"], auto_w_mae=a.get("w_mae"), nav2_w_mae=n.get("w_mae"), nav2_w_sign=n.get("w_sign_agree"),
                         ade=o["ade_m"], fde=o["fde_m"]))
json.dump(rows, open(f"{WORK}/eval_index.json", "w"), indent=1)
for r in rows:
    print(json.dumps(r))
